# Market Data Smoke Test

This starter notebook tests the market data helper adapted from `Portfolio_Optimization_2023`.

It verifies that we can:

- Import `MarketDataClient` from this repository.
- Pull daily OHLCV data from Yahoo Finance.
- Optionally pull from Alpaca when credentials are configured.
- Time a small multi-ticker request and inspect basic data quality.

The default date range is intentionally small so the test is quick and repeatable.

In [1]:
from pathlib import Path
import os
import sys
import time

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

src_path = PROJECT_ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from sentiment_ltr.data import MarketDataClient, MarketDataConfig

PROJECT_ROOT

PosixPath('/Users/armandoordoricadelatorre/Documents/U of T/PhD/PhD Research/Sentiment_learn_to_rank_paper')

## Yahoo Finance Smoke Test

Yahoo Finance is the easiest public-data path to verify first. This cell pulls a short SPY sample and prints elapsed time, shape, date range, and available columns.

In [2]:
config = MarketDataConfig(
    start="2023-01-01",
    end="2023-03-01",
    prefer_provider="yahoo",
)
client = MarketDataClient(config)

start_time = time.perf_counter()
spy = client.fetch("SPY")
elapsed = time.perf_counter() - start_time

print(f"Fetched SPY in {elapsed:.2f} seconds")
print(f"Shape: {spy.shape}")
print(f"Date range: {spy.index.min().date()} to {spy.index.max().date()}")
print(f"Columns: {list(spy.columns)}")

spy.head()

Fetched SPY in 0.57 seconds
Shape: (39, 10)
Date range: 2023-01-03 to 2023-02-28
Columns: ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'ticker', 'provider', 'Return', 'source_symbol']


Price,Open,High,Low,Close,Adj Close,Volume,ticker,provider,Return,source_symbol
date,,,,,,,,,,
2023-01-03,384.369995,386.429993,377.829987,380.820007,365.072052,74850700,SPY,yahoo,NaN,SPY
2023-01-04,383.179993,385.880005,380.000000,383.760010,367.890442,85934100,SPY,yahoo,0.007720,SPY
2023-01-05,381.720001,381.839996,378.760010,379.380005,363.691650,76970500,SPY,yahoo,-0.011413,SPY
2023-01-06,382.609985,389.250000,379.410004,388.079987,372.031799,104189600,SPY,yahoo,0.022932,SPY
2023-01-09,390.369995,393.700012,387.670013,387.859985,371.820953,73978100,SPY,yahoo,-0.000567,SPY


## Multi-Ticker Timing Test

This checks whether the helper can pull several liquid tickers quickly enough for iterative development. The current implementation fetches one ticker at a time, which is simple and reliable. If this becomes slow for hundreds of tickers, the next step is adding a vectorized Yahoo batch downloader or concurrent requests.

In [3]:
tickers = ["SPY", "AAPL", "MSFT", "AMZN", "GOOGL"]

start_time = time.perf_counter()
market_data = client.fetch_many(tickers)
elapsed = time.perf_counter() - start_time

summary = pd.DataFrame(
    {
        ticker: {
            "rows": len(df),
            "start": df.index.min(),
            "end": df.index.max(),
            "missing_close": int(df["Close"].isna().sum()) if "Close" in df else None,
            "provider": df["provider"].iloc[0] if not df.empty else None,
            "source_symbol": df["source_symbol"].iloc[0] if "source_symbol" in df and not df.empty else ticker,
        }
        for ticker, df in market_data.items()
    }
).T

print(f"Fetched {len(tickers)} tickers in {elapsed:.2f} seconds")
summary

Fetched 5 tickers in 1.45 seconds


,rows,start,end,missing_close,provider,source_symbol
SPY,39,2023-01-03 00:00:00,2023-02-28 00:00:00,0,yahoo,SPY
AAPL,39,2023-01-03 00:00:00,2023-02-28 00:00:00,0,yahoo,AAPL
MSFT,39,2023-01-03 00:00:00,2023-02-28 00:00:00,0,yahoo,MSFT
AMZN,39,2023-01-03 00:00:00,2023-02-28 00:00:00,0,yahoo,AMZN
GOOGL,39,2023-01-03 00:00:00,2023-02-28 00:00:00,0,yahoo,GOOGL


## Optional Alpaca Test

This cell uses Alpaca first only when `ALPACA_API_KEY` and `ALPACA_SECRET_KEY` are available in the environment or a local `.env` file. If credentials are missing or Alpaca fails for the request, the client falls back to Yahoo Finance.

In [4]:
alpaca_config = MarketDataConfig(
    start="2023-01-01",
    end="2023-03-01",
    prefer_provider="alpaca",
)
alpaca_client = MarketDataClient(alpaca_config)

has_alpaca_credentials = bool(os.getenv("ALPACA_API_KEY") and os.getenv("ALPACA_SECRET_KEY"))
print(f"Alpaca credentials configured: {has_alpaca_credentials}")

start_time = time.perf_counter()
alpaca_or_yahoo_spy = alpaca_client.fetch("SPY")
elapsed = time.perf_counter() - start_time

print(f"Fetched SPY in {elapsed:.2f} seconds")
print(f"Provider used: {alpaca_or_yahoo_spy['provider'].iloc[0]}")
alpaca_or_yahoo_spy.head()

Alpaca credentials configured: False
Fetched SPY in 0.01 seconds
Provider used: yahoo


Price,Open,High,Low,Close,Adj Close,Volume,ticker,provider,Return,source_symbol
date,,,,,,,,,,
2023-01-03,384.369995,386.429993,377.829987,380.820007,365.072052,74850700,SPY,yahoo,NaN,SPY
2023-01-04,383.179993,385.880005,380.000000,383.760010,367.890442,85934100,SPY,yahoo,0.007720,SPY
2023-01-05,381.720001,381.839996,378.760010,379.380005,363.691650,76970500,SPY,yahoo,-0.011413,SPY
2023-01-06,382.609985,389.250000,379.410004,388.079987,372.031799,104189600,SPY,yahoo,0.022932,SPY
2023-01-09,390.369995,393.700012,387.670013,387.859985,371.820953,73978100,SPY,yahoo,-0.000567,SPY
